# 프리킥 공(Ball) 추적 모델 자가개발 가이드 (Colab 환경)

이 노트북은 서버 비용 없이 구글 코랩(Colab)의 무료 GPU를 사용하여 자체 수집된 동영상(이미지) 데이터로 초경량 YOLO 모델을 훈련시키는 스크립트입니다.
완료 후 생성되는 `model.json` 및 `.bin` 파일을 앱의 `public/model` 디렉토리에 덮어쓰면 자가개발 모델이 앱에 적용됩니다.

In [ ]:
!pip install ultralytics tensorflowjs
import ultralytics
ultralytics.checks()

## 1. 데이터셋 가져오기
구글 드라이브를 연동하거나, 직접 압축 파일(dataset.zip)을 업로드하여 압축을 풉니다.

In [ ]:
import zipfile
import os

# 만약 로컬에서 dataset.zip을 업로드했다면 압축 풀기
if os.path.exists('dataset.zip'):
    with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print("데이터셋 압축 해제 완료")
else:
    print("dataset.zip 파일을 좌측 폴더 탭에 업로드 해주세요.")

## 2. 모델 훈련 (YOLOv8 Nano)
초경량 모델인 yolov8n 모델을 우리의 프리킥 데이터로 파인튜닝합니다.

In [ ]:
from ultralytics import YOLO

# 사전 훈련된 가벼운 모델 불러오기
model = YOLO('yolov8n.pt')

# 학습 시작 (epochs는 데이터량에 따라 50~100 권장)
# dataset/data.yaml 파일이 있어야 합니다.
results = model.train(data='dataset/data.yaml', epochs=50, imgsz=640, device=0)

## 3. 웹브라우저용 TensorFlow.js 형식으로 변환 (Export)
학습이 끝난 모델을 앱(모바일 웹)에서 사용할 수 있도록 tfjs 포맷으로 변환합니다.

In [ ]:
# best.pt를 tfjs로 추출
!yolo export model=runs/detect/train/weights/best.pt format=tfjs

print("변환이 완료되었습니다. 좌측 폴더에서 runs/detect/train/weights/best_web_model 폴더를 다운로드하세요.")